# P2 — Pipeline de Datos Automatizado

**RA/CE**: RA3b, RA3c
**Fase**: F2 — Pipeline de Datos
**Teoría asociada**: `01-teoria/03_pipeline_datos.md`

---

## Objetivos de Aprendizaje

1. Diseñar un pipeline ETL reproducible con scripts independientes
2. Versionar datasets con DVC y conectar versiones de datos con versiones de código
3. Implementar validación de esquemas con pandera
4. Evaluar las características de diferentes sistemas de conexión (manual vs. automatizado)

## Prerrequisitos

- [ ] Concepto de ETL básico (visto en prácticas UD6)
- [ ] Conocimiento de DVC (visto en UD5: `05-cloud-mlops/01-teoria/01-feature-store.md`)
- [ ] Finalizada la práctica P1 (tienes las funciones de carga)

## Contexto

En la práctica P1 generaste funciones para cargar y limpiar datos. Ahora vamos
a convertirlas en un **pipeline reproducible, versionado y validado** que
produzca los datos de entrenamiento que usarás en P3 (MLflow).

---
## Parte 1: Verifica tu entorno

In [ ]:
import sys, os, subprocess, json

HAS_DEPS = True
missing = []

try:
    import pandas as pd
    import numpy as np
    import pandera as pa
    from sklearn.model_selection import train_test_split
    from sklearn.feature_extraction.text import TfidfVectorizer
    print('✅ Python dependencies OK')
except ImportError as e:
    HAS_DEPS = False
    mod = str(e).split("'")[1]
    missing.append(mod)
    print(f'❌ Missing: {mod}')

if not HAS_DEPS:
    print(f'\nInstall: pip install {" ".join(missing)}')
else:
    print('\n✅ Environment ready')

---
## Parte 2: Crear la estructura del pipeline

Vamos a organizar el pipeline en scripts independientes dentro de
`07-convergencia-herramientas/02-ejemplos/pipeline_completo/`:

```
pipeline_completo/
├── data/
│   ├── raw/           # Datos originales (versionados con DVC)
│   ├── processed/     # Datos limpios
│   └── features/      # Features listos para entrenar
├── scripts/
│   ├── extract.py     # Extracción de datos
│   ├── clean.py       # Limpieza y validación
│   ├── features.py    # Feature engineering
│   └── split.py       # Train/test split
├── tests/
│   └── test_pipeline.py  # Tests unitarios
└── requirements.txt
```

Ejecuta la celda siguiente para crear la estructura:

In [ ]:
import pathlib

# Base path for our pipeline example
BASE = pathlib.Path("..") / "02-ejemplos" / "pipeline_completo"
BASE = BASE.resolve()

dirs = [
    BASE / "data" / "raw",
    BASE / "data" / "processed",
    BASE / "data" / "features",
    BASE / "scripts",
    BASE / "tests",
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f'📁 {d.relative_to(BASE.parent.parent)}/')

print(f'\n✅ Structure created at: {BASE}')

---
## Parte 3: Script de extracción

Crea un script `extract.py` que cargue datos desde múltiples fuentes.
El código debe ser **independiente** (no depende del notebook) y **reproducible**.

In [ ]:
# Escribe el contenido del script extract.py
# Usa el asistente de IA para generarlo o escríbelo tú mismo

EXTRACT_SCRIPT = '''"""
Pipeline de datos - Extraccion
RA3b: Sistemas que facilitan la conexion tecnologica
"""
import pandas as pd
import pathlib
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def extract_from_csv(filepath: str) -> pd.DataFrame:
    """Extract ticket data from CSV file.
    
    Args:
        filepath: Path to CSV file
    
    Returns:
        DataFrame with raw ticket data
    """
    logger.info(f"Extracting from {filepath}")
    return pd.read_csv(filepath, parse_dates=['created_at'])


def extract_from_api(url: str, params: dict = None) -> pd.DataFrame:
    """Extract ticket data from a REST API.
    
    Args:
        url: API endpoint URL
        params: Optional query parameters
    Returns:
        DataFrame with ticket data from API
    """
    import requests
    logger.info(f"Extracting from API: {url}")
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return pd.DataFrame(resp.json())


def run(input_path: str, output_path: str) -> None:
    """Run the extract step."""
    df = extract_from_csv(input_path)
    pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    logger.info(f"Extracted {len(df)} rows to {output_path}")


if __name__ == "__main__":
    import sys
    run(sys.argv[1], sys.argv[2])
'''

script_path = BASE / "scripts" / "extract.py"
script_path.write_text(EXTRACT_SCRIPT)
print(f'✅ Created {script_path.relative_to(BASE.parent.parent)}')

### Ejercicio 3.1: Crear datos de prueba

Genera un dataset sintético de incidencias técnicas para probar el pipeline.

In [ ]:
# Generar dataset sintetico
import numpy as np

np.random.seed(42)
n_samples = 100

categories = ['infraestructura', 'seguridad', 'rendimiento', 'software', 'hardware']
priorities = ['alta', 'media', 'baja']
statuses = ['abierto', 'en_curso', 'cerrado', 'reabierto']

descriptions = [
    # infraestructura
    'El servidor principal no responde a las peticiones HTTP',
    'La conexion de red se cae intermitentemente cada 30 minutos',
    'El balanceador de carga no distribuye el trafico correctamente',
    'El firewall bloquea puertos necesarios para la aplicacion',
    # seguridad
    'Intento de acceso no autorizado detectado en el panel admin',
    'El certificado SSL ha expirado y los usuarios ven advertencia',
    'Posible fuga de datos en el endpoint de usuarios',
    # rendimiento
    'Las consultas a la base de datos tardan mas de 30 segundos',
    'El tiempo de respuesta de la API supera los 5 segundos',
    'La aplicacion web se congela al cargar el dashboard',
    # software
    'Error 500 al intentar guardar un formulario',
    'La interfaz de usuario no muestra los datos actualizados',
    'El proceso de exportacion falla sin mensaje de error',
    # hardware
    'El disco duro del servidor de base de datos esta al 95%% de capacidad',
    'La memoria RAM del nodo de procesamiento se satura con carga normal',
    'El ventilador de la GPU emite ruido excesivo y temperatura alta'
]

data = []
for i in range(n_samples):
    desc = np.random.choice(descriptions)
    cat = next(c for c in categories if desc in [
        d for d in descriptions if categories[descriptions.index(d) // 4] == c
    ] if False)  # simplified: just pick random
    # Simplified category assignment
    idx = descriptions.index(desc) if desc in descriptions else 0
    if idx < 4: cat = 'infraestructura'
    elif idx < 7: cat = 'seguridad'
    elif idx < 10: cat = 'rendimiento'
    elif idx < 13: cat = 'software'
    else: cat = 'hardware'
    
    data.append({
        'ticket_id': i + 1,
        'created_at': pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(np.random.randint(0, 365))),
        'description': desc,
        'category': cat,
        'priority': np.random.choice(priorities, p=[0.2, 0.5, 0.3]),
        'status': np.random.choice(statuses, p=[0.3, 0.2, 0.4, 0.1])
    })

df_raw = pd.DataFrame(data)
raw_path = BASE / "data" / "raw" / "tickets.csv"
df_raw.to_csv(raw_path, index=False)
print(f'✅ Created synthetic dataset: {len(df_raw)} tickets')
print(df_raw.head())
print(f'\nCategories: {df_raw["category"].value_counts().to_dict()}')

---
## Parte 4: Script de limpieza con validación

Crea un script `clean.py` que limpie los datos y valide el esquema.

In [ ]:
CLEAN_SCRIPT = '''"""
Pipeline de datos - Limpieza y validacion
RA3c: Evaluacion de caracteristicas de sistemas de conexion
"""
import pandas as pd
import pandera as pa
from pandera import Check, Column, DataFrameSchema
import pathlib
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# Definir el esquema de validacion
SCHEMA = DataFrameSchema({
    "ticket_id": Column(int, Check.unique(), nullable=False),
    "created_at": Column(pd.DatetimeTZDtype(tz="UTC"), nullable=True),
    "description": Column(str, 
        Check.str_length(1, 10000, ignore_na=True),
        nullable=False
    ),
    "category": Column(str, Check.isin([
        "infraestructura", "seguridad", "rendimiento",
        "software", "hardware"
    ])),
    "priority": Column(str, Check.isin(["alta", "media", "baja"])),
    "status": Column(str, Check.isin([
        "abierto", "en_curso", "cerrado", "reabierto"
    ])),
}, strict=True)


def clean_tickets(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and validate ticket data."""
    initial = len(df)
    
    # Remove rows with empty description
    df = df[df['description'].notna() & (df['description'].str.strip() != '')]
    
    # Normalize text fields
    df['priority'] = df['priority'].str.lower().str.strip()
    df['category'] = df['category'].str.lower().str.strip()
    df['status'] = df['status'].str.lower().str.strip()
    
    # Add computed columns
    df['text_length'] = df['description'].str.len()
    df['cleaned_at'] = pd.Timestamp.now(tz='UTC')
    
    logger.info(f"Cleaned: {initial} -> {len(df)} rows")
    return df


def validate(df: pd.DataFrame) -> bool:
    """Validate DataFrame against schema."""
    try:
        SCHEMA.validate(df, lazy=True)
        logger.info("Validation PASSED")
        return True
    except pa.errors.SchemaErrors as e:
        logger.error(f"Validation FAILED: {e}")
        return False


def run(input_path: str, output_path: str) -> None:
    """Run the clean step."""
    df = pd.read_csv(input_path, parse_dates=['created_at'])
    df = clean_tickets(df)
    if not validate(df):
        raise ValueError("Data validation failed - stopping pipeline")
    pathlib.Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    logger.info(f"Saved clean data to {output_path}")


if __name__ == "__main__":
    import sys
    run(sys.argv[1], sys.argv[2])
'''

script_path = BASE / "scripts" / "clean.py"
script_path.write_text(CLEAN_SCRIPT)
print(f'✅ Created {script_path.relative_to(BASE.parent.parent)}')

---
## Parte 5: Feature engineering y split

Crea los scripts de transformación de features y división train/test.

In [ ]:
FEATURES_SCRIPT = '''"""
Pipeline de datos - Feature Engineering
"""
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import pathlib
import pickle
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def create_features(df: pd.DataFrame, fit: bool = True,
                    vectorizer_path: str = None) -> tuple:
    """Create TF-IDF features from ticket descriptions.
    
    Args:
        df: Cleaned DataFrame
        fit: If True, fit the vectorizer; if False, load existing
        vectorizer_path: Path to saved vectorizer (load if fit=False,
                        save if fit=True)
    Returns:
        X (sparse matrix), y (Series), vectorizer
    """
    if fit:
        vectorizer = TfidfVectorizer(
            max_features=5000,
            stop_words='english',
            min_df=2,
            max_df=0.95
        )
        X = vectorizer.fit_transform(df['description'])
        if vectorizer_path:
            pathlib.Path(vectorizer_path).parent.mkdir(parents=True, exist_ok=True)
            with open(vectorizer_path, 'wb') as f:
                pickle.dump(vectorizer, f)
            logger.info(f"Vectorizer saved to {vectorizer_path}")
    else:
        with open(vectorizer_path, 'rb') as f:
            vectorizer = pickle.load(f)
        X = vectorizer.transform(df['description'])
    
    y = df['category']
    return X, y, vectorizer


def run(input_path: str, output_X_path: str, output_y_path: str,
        vectorizer_path: str) -> None:
    df = pd.read_csv(input_path)
    X, y, _ = create_features(df, fit=True, vectorizer_path=vectorizer_path)
    pathlib.Path(output_X_path).parent.mkdir(parents=True, exist_ok=True)
    with open(output_X_path, 'wb') as f:
        pickle.dump(X, f)
    y.to_csv(output_y_path, index=False, header=['category'])
    logger.info(f"Features: {X.shape}, saved to {output_X_path}")


if __name__ == "__main__":
    import sys
    run(sys.argv[1], sys.argv[2], sys.argv[3], sys.argv[4])
'''

script_path = BASE / "scripts" / "features.py"
script_path.write_text(FEATURES_SCRIPT)
print(f'✅ Created {script_path.relative_to(BASE.parent.parent)}')

In [ ]:
SPLIT_SCRIPT = '''"""
Pipeline de datos - Train/Test Split
"""
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
import pathlib
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


def run(X_path: str, y_path: str, output_dir: str,
        test_size: float = 0.2, random_state: int = 42) -> None:
    with open(X_path, 'rb') as f:
        X = pickle.load(f)
    y = pd.read_csv(y_path)['category']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state,
        stratify=y
    )
    
    out = pathlib.Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    
    for name, data in [('X_train', X_train), ('X_test', X_test),
                       ('y_train', y_train), ('y_test', y_test)]:
        with open(out / f'{name}.pkl', 'wb') as f:
            pickle.dump(data, f)
    
    logger.info(f"Split: {X_train.shape[0]} train, {X_test.shape[0]} test")


if __name__ == "__main__":
    import sys
    run(sys.argv[1], sys.argv[2], sys.argv[3])
'''

script_path = BASE / "scripts" / "split.py"
script_path.write_text(SPLIT_SCRIPT)
print(f'✅ Created {script_path.relative_to(BASE.parent.parent)}')

---
## Parte 6: Ejecutar el pipeline completo

Ahora ejecuta los scripts en orden para procesar los datos de principio a fin.

In [ ]:
%%bash
# Ejecutar el pipeline paso a paso
BASE="../../02-ejemplos/pipeline_completo"

echo "=== Step 1: Extract ==="
python "$BASE/scripts/extract.py" "$BASE/data/raw/tickets.csv" "$BASE/data/processed/tickets_raw.csv"

echo ""
echo "=== Step 2: Clean ==="
python "$BASE/scripts/clean.py" "$BASE/data/processed/tickets_raw.csv" "$BASE/data/processed/tickets_clean.csv"

echo ""
echo "=== Step 3: Features ==="
mkdir -p "$BASE/data/features"
python "$BASE/scripts/features.py" "$BASE/data/processed/tickets_clean.csv" "$BASE/data/features/X.pkl" "$BASE/data/features/y.csv" "$BASE/models/vectorizer.pkl"

echo ""
echo "=== Step 4: Split ==="
mkdir -p "$BASE/data/splits"
python "$BASE/scripts/split.py" "$BASE/data/features/X.pkl" "$BASE/data/features/y.csv" "$BASE/data/splits/"

echo ""
echo "=== Pipeline complete ==="
ls -la "$BASE/data/splits/"

---
## Parte 7: DVC — Versionado de datos

Inicializa DVC en el directorio del pipeline y versiona los datos.

In [ ]:
%%bash
# Inicializar DVC en el directorio del pipeline
cd ../../02-ejemplos/pipeline_completo

# Verificar si DVC esta instalado
if ! command -v dvc &> /dev/null; then
    echo "DVC no instalado. Instalando..."
    pip install dvc -q
fi

dvc init 2>/dev/null || echo "DVC ya inicializado"

# Versionar datos con DVC
dvc add data/raw/tickets.csv --quiet 2>/dev/null
echo "✅ Datos versionados con DVC"
cat data/raw/tickets.csv.dvc

### Ejercicio 7.1: Modificar datos y versionar

Añade 10 registros más al dataset y versiona la nueva versión. ¿Qué cambia en el
archivo `.dvc`? ¿Cómo sabrías qué versión de datos usaste para un modelo concreto?

In [ ]:
# Ejercicio: anadir mas datos y versionar con DVC
new_tickets = pd.DataFrame({
    'ticket_id': range(101, 111),
    'description': [
        'Error al procesar pago en la pasarela',
        'La aplicacion movil no sincroniza datos',
        'El servicio de notificaciones no envia emails',
        'Tiempo de espera en la carga de imagenes',
        'Error de permisos al acceder a archivos compartidos',
        'El chatbot no entiende preguntas en espanol',
        'La busqueda de tickets devuelve resultados incompletos',
        'El reporte semanal no se genera automaticamente',
        'La integracion con Slack falla intermitentemente',
        'Error al exportar datos a Excel con formato incorrecto'
    ],
    'category': ['software', 'software', 'infraestructura', 'rendimiento',
                 'seguridad', 'software', 'rendimiento', 'software',
                 'infraestructura', 'software'],
    'priority': ['alta', 'media', 'alta', 'media', 'alta',
                 'baja', 'media', 'baja', 'media', 'baja'],
    'status': ['abierto'] * 10
})

df_existing = pd.read_csv(raw_path)
df_updated = pd.concat([df_existing, new_tickets], ignore_index=True)
df_updated.to_csv(raw_path, index=False)
print(f'✅ Dataset actualizado: {len(df_updated)} registros')
print(f'\nAhora ejecuta en terminal:')
print(f'  cd 02-ejemplos/pipeline_completo')
print(f'  dvc add data/raw/tickets.csv')

---
## Parte 8: Ejercicios para el alumno

### Ejercicio A: Validación de datos corruptos

Introduce datos inválidos en el CSV (una categoría inexistente, un ticket_id
duplicado, una prioridad mal escrita) y ejecuta `clean.py`. ¿Qué ocurre?
¿Cómo responde la validación con pandera?

**Anota los resultados**:
```
______________________________________________________________
______________________________________________________________
```

### Ejercicio B: Evaluación de sistemas (RA3c)

Compara el pipeline automatizado que acabas de construir con el proceso manual
que usarías en un notebook. Completa la tabla:

| Criterio | Notebook manual | Pipeline automatizado |
|----------|----------------|----------------------|
| Reproducibilidad | | |
| Tolerancia a errores | | |
| Facilidad de cambio | | |
| Trazabilidad | | |
| Tiempo de ejecución | | |

**¿En qué casos usarías cada enfoque?**
```
______________________________________________________________
______________________________________________________________
```

### Ejercicio C: Integración con F1

Vuelve a la función `load_tickets` que generaste en P1. ¿Qué diferencias hay
entre esa función y el script `extract.py` que acabas de crear? ¿Qué ventajas
tiene el script independiente frente a la función embebida en el notebook?

```
______________________________________________________________
______________________________________________________________
```

---
## Referencias

### Teoría
- `01-teoria/03_pipeline_datos.md` — Guía teórica de F2
- `01-teoria/01_gap_produccion.md` — Contexto del gap

### UD5/UD6
- `05-cloud-mlops/01-teoria/01-feature-store.md` — Feature store
- `05-cloud-mlops/01-teoria/03-mlops.md` — MLOps pipeline
- `06-llm-agentes/03-practicas/99_herramientas_ia_integradas.ipynb` — ETL básico

### Documentación
- [DVC Documentation](https://dvc.org/doc)
- [Pandera Docs](https://pandera.readthedocs.io/)